In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

prices = yf.download(["OMER", "SPY"], start="2024-09-01", end="2026-02-01",
                     auto_adjust=True, progress=False)["Close"]
returns = prices.pct_change().dropna()
returns.tail()

Ticker,OMER,SPY
Date,,
2026-01-26,-0.033529,0.005078
2026-01-27,0.098873,0.003984
2026-01-28,-0.067877,-0.000101
2026-01-29,0.011854,-0.001984
2026-01-30,-0.021757,-0.002983


In [3]:
import statsmodels.api as sm

day0 = pd.Timestamp("2025-12-24")
pos = returns.index.get_loc(day0)

est = returns.iloc[pos - 250 : pos - 30]   # days -250 through -31
X = sm.add_constant(est["SPY"])
model = sm.OLS(est["OMER"], X).fit()

alpha, beta = model.params["const"], model.params["SPY"]
print(len(est), "estimation days")
print(f"alpha = {alpha:.5f}, beta = {beta:.2f}, R-squared = {model.rsquared:.3f}")
print(f"beta standard error = {model.bse['SPY']:.2f}")

220 estimation days
alpha = 0.00163, beta = 1.30, R-squared = 0.020
beta standard error = 0.62


In [4]:
event = returns.iloc[pos - 5 : pos + 11].copy()
event["day"] = range(-5, 11)
event["expected"] = alpha + beta * event["SPY"]
event["abnormal"] = event["OMER"] - event["expected"]
event[["day", "OMER", "SPY", "expected", "abnormal"]].round(4)

Ticker,day,OMER,SPY,expected,abnormal
Date,,,,,
2025-12-17,-5,0.0068,-0.0110,-0.0127,0.0196
2025-12-18,-4,0.0374,0.0076,0.0115,0.0259
2025-12-19,-3,0.0480,0.0091,0.0135,0.0346
2025-12-22,-2,-0.0094,0.0062,0.0098,-0.0191
2025-12-23,-1,-0.0799,0.0046,0.0076,-0.0875
2025-12-24,0,0.7554,0.0035,0.0062,0.7492
2025-12-26,1,0.0228,-0.0001,0.0015,0.0213
2025-12-29,2,0.0547,-0.0036,-0.0030,0.0578
2025-12-30,3,0.0235,-0.0012,0.0000,0.0235


In [5]:
car_short = event.loc[event["day"].between(-1, 1), "abnormal"].sum()
car_long = event.loc[event["day"].between(0, 5), "abnormal"].sum()
print(f"CAR[-1,+1] = {car_short:.2%}")
print(f"CAR[0,+5]  = {car_long:.2%}")

CAR[-1,+1] = 68.30%
CAR[0,+5]  = 82.22%


In [6]:
def event_study(ticker, day0):
    day0 = pd.Timestamp(day0)
    px = yf.download([ticker, "SPY"], start=day0 - pd.Timedelta(days=400),
                     end=day0 + pd.Timedelta(days=30), auto_adjust=True, progress=False)["Close"]
    r = px.pct_change().dropna()

    pos = r.index.searchsorted(day0)          # first trading day on or after day0
    if pos < 250 or pos + 5 >= len(r):
        return None                           # not enough data

    est = r.iloc[pos - 250 : pos - 30]
    model = sm.OLS(est[ticker], sm.add_constant(est["SPY"])).fit()
    a, b = model.params["const"], model.params["SPY"]

    win = r.iloc[pos - 1 : pos + 6].copy()   # days -1 through +5
    win["day"] = range(-1, 6)
    win["ar"] = win[ticker] - (a + b * win["SPY"])

    return {"ticker": ticker, "day0": r.index[pos].date(), "beta": round(b, 2),
            "r2": round(model.rsquared, 3),
            "car_m1_p1": win.loc[win["day"].between(-1, 1), "ar"].sum(),
            "car_0_p5": win.loc[win["day"].between(0, 5), "ar"].sum()}

event_study("OMER", "2025-12-24")

{'ticker': 'OMER',
 'day0': datetime.date(2025, 12, 24),
 'beta': np.float64(1.3),
 'r2': np.float64(0.02),
 'car_m1_p1': np.float64(0.6829991252582485),
 'car_0_p5': np.float64(0.8222003639265277)}

In [7]:
events = pd.DataFrame({
    "drug":   ["Yartemlea", "Komzifti", "Redemplo", "Palsonify", "Dawnzera"],
    "ticker": ["OMER", "KURA", "ARWR", "CRNX", "IONS"],
    "day0":   ["2025-12-24", "2025-11-13", "2025-11-18", "2025-09-26", "2025-08-21"],
})

results = []
for _, e in events.iterrows():
    try:
        res = event_study(e["ticker"], e["day0"])
    except Exception as err:
        print(e["ticker"], "failed:", err)
        continue
    if res is None:
        print(e["ticker"], "skipped: not enough price data")
        continue
    res["drug"] = e["drug"]
    results.append(res)

pd.DataFrame(results)

$CRNX: Data doesn't exist for startDate = 1724299200, endDate = 1761451200

1 Failed download:
['CRNX']: Data doesn't exist for startDate = 1724299200, endDate = 1761451200


CRNX skipped: not enough price data


,ticker,day0,beta,r2,car_m1_p1,car_0_p5,drug
0,OMER,2025-12-24,1.30,0.020,0.682999,0.822200,Yartemlea
1,KURA,2025-11-13,0.90,0.068,0.070941,0.087843,Komzifti
2,ARWR,2025-11-18,1.72,0.219,-0.041117,0.119336,Redemplo
3,IONS,2025-08-21,0.87,0.172,-0.000600,-0.014986,Dawnzera
